In [1]:
import sys
!{sys.executable} -m pip install -e .

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///sdf/group/neutrino/pgranger/larnd-sim-jax
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for larndsim (pyproject.toml) ... done
  Created wheel for larndsim: filename=larndsim-0.1.dev283+ga1569e19c.d20251031-py3-none-any.whl size=10155 sha256=f66565a95921e38d97878d0943b7e65b9f7dcf434fa3ab25c54d83f29868056e
  Stored in directory: /lscratch/pgranger/tmp/pip-ephem-wheel-cache-u72p3r7_/wheels/e7/e6/52/708c2eabf9d963fabdda3635808ffba6fe643aaaafd1316c9c
Successfully built larndsim
  Attempting uninstall: larndsim
    Found existing installation: larndsim 0.1.dev283+ga1569e19c.d20251030
    Uninstalling larndsim-0.1.dev283+ga1569e19c.d20251030:
      Successfully uninstalled lar

In [1]:
import jax
import jax.numpy as jnp
from larndsim.sim_jax import simulate_drift_new, prepare_tracks
from larndsim.consts_jax import build_params_class, load_detector_properties, load_lut

In [2]:
from optimize.dataio import pad_sequence

to_propagate = ('Ab', 'kb', 'long_diff', 'tran_diff', 'eField', 'lifetime', 'shift_x', 'shift_y', 'shift_z')
# to_propagate = ('shift_x', 'shift_y', 'shift_z')
#Running some fit
Params = build_params_class(to_propagate)
ref_params = load_detector_properties(Params, "./src/larndsim/detector_properties/module0.yaml",
                                "./src/larndsim/pixel_layouts/multi_tile_layout-2.4.16_v4.yaml")
ref_params = ref_params.replace(Ab=0.8, kb=0.0486, tran_diff=4.0e-6, long_diff=8.8e-6, number_pix_neighbors=0, signal_length=100, electron_sampling_resolution=5e-3, eField=0.5, lifetime=3e3, time_padding=190)
ref_params = ref_params.replace(time_window=ref_params.signal_length)
# ref_params = ref_params.replace(RESET_NOISE_CHARGE=0., UNCORRELATED_NOISE_CHARGE=0., lifetime=1e10, long_diff=0., tran_diff=0.)
# ref_params = ref_params.replace(RESET_NOISE_CHARGE=0., UNCORRELATED_NOISE_CHARGE=0.)

response_template, ref_params = load_lut("src/larndsim/detector_properties/response_44.npy", ref_params)

tracks_files = [
    './prepared_data/input_1.h5',
    './prepared_data/input_2.h5',
    # './prepared_data/input_3.h5',
    # './prepared_data/input_4.h5'
]
tracks = None
for tracks_file in tracks_files:
    tracks_loc, fields, _ = prepare_tracks(ref_params, tracks_file, False)
    if tracks is None:
        tracks = tracks_loc
    else:
        tracks = jnp.concatenate([tracks, tracks_loc])

INFO:larndsim.consts_jax:Loading response from numpy array
Could not load symbol cuFuncGetName. Error: /.singularity.d/libs/libcuda.so.1: undefined symbol: cuFuncGetName


In [3]:
# Lower the jitted function to get a "compiled" object
compiled_fn = simulate_drift_new.lower(ref_params, tracks, fields).compile()

# Get the cost analysis dictionary
cost = compiled_fn.cost_analysis()

# Print the results in a readable format
print(f"FLOPs: {cost['flops']:,}")
print(f"Bytes Accessed: {cost['bytes accessed']:,}")
print(f"Transcendentals: {cost['transcendentals']:,}")

FLOPs: 164,172,576.0
Bytes Accessed: 101,335,928.0
Transcendentals: 1,113,576.0


In [3]:
# Lower the jitted function to get a "compiled" object
compiled_fn = simulate_drift_new.lower(ref_params, tracks, fields).compile()

# Get the cost analysis dictionary
cost = compiled_fn.cost_analysis()

# Print the results in a readable format
print(f"FLOPs: {cost['flops']:,}")
print(f"Bytes Accessed: {cost['bytes accessed']:,}")
print(f"Transcendentals: {cost['transcendentals']:,}")

FLOPs: 164,172,576.0
Bytes Accessed: 101,335,928.0
Transcendentals: 1,113,576.0
